<a href="https://colab.research.google.com/github/Sizwe100/Python-Random-Forest/blob/main/OBJECTIVE1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install and import required libraries
!pip install earthengine-api geopandas rasterio -q

import ee
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import time

In [3]:
# 2. Authenticate and Initialize Google Earth Engine
project_id = 'ee-sizwepmthimkhulu' # <-- REPLACE WITH YOUR GEE PROJECT ID

try:
    ee.Initialize(project=project_id)
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project=project_id)

In [7]:
# Mount Google Drive to access your files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Once Google Drive is mounted, we need to locate your CSV file. By default, your main Drive folder is at `/content/drive/MyDrive/`.

If your CSV is in a specific folder, you should update the path. For example, if it's in a folder called `Colab Notebooks`, the path will be:
`"/content/drive/MyDrive/Colab Notebooks/Ximenia_afra.csv"`.

Run the cell below to find and set the correct file path.

In [8]:
import os

# We will search your Google Drive to automatically find the exact path of 'Ximenia_afra.csv'
found_path = None
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "Ximenia_afra.csv" in files:
        found_path = os.path.join(root, "Ximenia_afra.csv")
        print(f"🎉 Found your file at: {found_path}")
        break

if found_path:
    csv_path = found_path
else:
    print("❌ Could not find 'Ximenia_afra.csv' in your Google Drive.")
    print("Please make sure the file name is spelled exactly 'Ximenia_afra.csv' inside your Drive.")

🎉 Found your file at: /content/drive/MyDrive/CLIMATE_VARIABLES/Ximenia_afra.csv


In [11]:
# 3. Load your CSV file
# The path was automatically discovered from your Google Drive in the previous step
try:
    df = pd.read_csv(csv_path)
except NameError:
    # Fallback in case the search cell was not run
    csv_path = "/content/drive/MyDrive/CLIMATE_VARIABLES/Ximenia_afra.csv"
    df = pd.read_csv(csv_path)

# Clean data: drop rows missing coordinates
df = df.dropna(subset=['longitude', 'latitude']) # <-- ADJUST IF COLUMNS ARE NAMED 'X'/'Y'

print(f"Loaded {len(df)} valid coordinate points.")

# 4. Convert Pandas DataFrame to Earth Engine FeatureCollection
features = []
for idx, row in df.iterrows():
    # Keep all your original CSV attributes
    properties = row.to_dict()
    # Create GEE geometry point
    geom = ee.Geometry.Point([row['longitude'], row['latitude']])
    features.append(ee.Feature(geom, properties))

points_ee = ee.FeatureCollection(features)

# 5. Load ERA5-Land Monthly Temperature dataset (2017 to late 2026)
era5 = (
    ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
    .filterDate("2017-01-01", "2026-10-31")
    .select("temperature_2m")
)

# 6. Function to map over collection and extract values
def extract_temp(image):
    # Cast generic element to an Earth Engine Image explicitly
    img = ee.Image(image)

    # Convert Kelvin to Celsius
    celsius_img = ee.Image(img.subtract(273.15)).copyProperties(img, ['system:time_start'])
    date_str = ee.Date(img.get('system:time_start')).format('YYYY-MM')

    # Sample regions at points (cast celsius_img to ee.Image explicitly to avoid Element AttributeError)
    sampled = ee.Image(celsius_img).sampleRegions(
        collection=points_ee,
        properties=list(df.columns), # Retains all your original CSV columns
        scale=9000
    )

    # Set the year-month string on every sampled point
    return sampled.map(lambda f: f.set('Year_Month', date_str))

# Run the extraction and flatten into a single FeatureCollection
print("Extracting temperature time-series from GEE datasets... Please wait.")
time_series_fc = era5.map(extract_temp).flatten()

# 7. Export straight to your Google Drive as a CSV
task = ee.batch.Export.table.toDrive(
    collection=time_series_fc,
    description='Ximenia_Caffra_Temp_Extract_2017_2026',
    fileFormat='CSV'
)
task.start()

print("⚡ Success! The extraction task has been submitted to Earth Engine.")
print("Check your Google Drive in a few minutes for 'Ximenia_Caffra_Temp_Extract_2017_2026.csv'.")

Loaded 220 valid coordinate points.
Extracting temperature time-series from GEE datasets... Please wait.
⚡ Success! The extraction task has been submitted to Earth Engine.
Check your Google Drive in a few minutes for 'Ximenia_Caffra_Temp_Extract_2017_2026.csv'.


In [12]:
# Load Landsat 8 Surface Reflectance Tier 1 Collection
# Covers 2013-present, perfect for your 2017-2026 analysis window
landsat = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterDate("2017-01-01", "2026-10-31")
)

# Helper function to compute NDVI, NDWI, and EVI for each Landsat image
def calculate_indices(image):
    img = ee.Image(image)

    # Landsat 8 Bands:
    # SR_B5 = NIR, SR_B4 = Red, SR_B3 = Green, SR_B6 = SWIR1, SR_B2 = Blue
    nir = img.select('SR_B5').multiply(0.0000275).add(-0.2)      # Scale factor applied
    red = img.select('SR_B4').multiply(0.0000275).add(-0.2)
    green = img.select('SR_B3').multiply(0.0000275).add(-0.2)
    swir = img.select('SR_B6').multiply(0.0000275).add(-0.2)
    blue = img.select('SR_B2').multiply(0.0000275).add(-0.2)

    # 1. NDVI = (NIR - Red) / (NIR + Red)
    ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')

    # 2. NDWI = (Green - NIR) / (Green + NIR)  - measures water stress
    ndwi = green.subtract(nir).divide(green.add(nir)).rename('NDWI')

    # 3. EVI = 2.5 * ((NIR - Red) / (NIR + 6 * Red - 7.5 * Blue + 1))
    evi = img.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1.0))',
        {
            'NIR': nir,
            'RED': red,
            'BLUE': blue
        }
    ).rename('EVI')

    # Return the image with the calculated indices added as bands
    return img.addBands([ndvi, ndwi, evi]).copyProperties(img, ['system:time_start'])

# Apply the calculation over our collection
processed_landsat = landsat.map(calculate_indices)

# Calculate Historical Max and Min NDVI per pixel for VCI calculation
ndvi_col = processed_landsat.select('NDVI')
ndvi_max = ndvi_col.max()
ndvi_min = ndvi_col.min()

# Function to extract monthly averaged indices at your points
def extract_vegetation_metrics(image):
    img = ee.Image(image)
    date_str = ee.Date(img.get('system:time_start')).format('YYYY-MM')

    # 4. Calculate VCI: VCI = 100 * (NDVI - NDVI_min) / (NDVI_max - NDVI_min)
    current_ndvi = img.select('NDVI')
    vci = current_ndvi.subtract(ndvi_min).divide(ndvi_max.subtract(ndvi_min)).multiply(100).rename('VCI')

    # Combine all target bands
    metrics_img = img.select(['NDVI', 'NDWI', 'EVI']).addBands(vci)

    # Sample at your coordinates
    sampled = metrics_img.sampleRegions(
        collection=points_ee,
        properties=list(df.columns),
        scale=30 # Landsat 30m resolution
    )

    return sampled.map(lambda f: f.set('Year_Month', date_str))

# Map over the collection, extract metrics, and flatten results
print("Extracting NDVI, NDWI, EVI, and VCI... This may take a moment.")
veg_time_series_fc = processed_landsat.map(extract_vegetation_metrics).flatten()

# Submit Export Task to Google Drive
veg_task = ee.batch.Export.table.toDrive(
    collection=veg_time_series_fc,
    description='Ximenia_Caffra_Vegetation_Indices_2017_2026',
    fileFormat='CSV'
)
veg_task.start()

print("⚡ Success! The extraction task has been submitted to Earth Engine.")
print("Check your Google Drive shortly for 'Ximenia_Caffra_Vegetation_Indices_2017_2026.csv'")

Extracting NDVI, NDWI, EVI, and VCI... This may take a moment.


⚡ Success! The extraction task has been submitted to Earth Engine.
Check your Google Drive shortly for 'Ximenia_Caffra_Vegetation_Indices_2017_2026.csv'


In [13]:
import time

# Get a list of all recent tasks
tasks = ee.batch.Task.list()

print("=== YOUR ACTIVE EARTH ENGINE EXPORT TASKS ===\n")
active_tasks = False
for t in tasks[:5]: # Show the 5 most recent tasks
    status = t.status()
    state = status.get('state')
    description = status.get('description')
    task_id = status.get('id')

    print(f"Task: {description}")
    print(f"ID: {task_id}")
    print(f"Status: {state}")
    if state in ['READY', 'RUNNING']:
        active_tasks = True
        # If it's running, show attempt progress if available
        if 'progress' in status:
            print(f"Progress: {status['progress']}")
    print("-" * 40)

if active_tasks:
    print("\n⏳ Your tasks are still processing on GEE servers. Run this cell again in a minute to refresh the status.")
else:
    print("\n✅ No active running tasks found in the top list. If they marked as COMPLETED, check your Google Drive 'MyDrive' root folder!")

=== YOUR ACTIVE EARTH ENGINE EXPORT TASKS ===

Task: Ximenia_Caffra_Vegetation_Indices_2017_2026
ID: YBQU2LJFPSTCX7VNKVTWFWOV
Status: RUNNING
----------------------------------------
Task: Ximenia_Caffra_Temp_Extract_2017_2026
ID: LHJOY7E2THDJTUJFKV42WG4I
Status: COMPLETED
----------------------------------------
Task: Ingest table: "projects/ee-sizwepmthimkhulu/assets/observations-763984"
ID: BKX33K5YDIOL3DXTLCM6OCFS
Status: COMPLETED
----------------------------------------

⏳ Your tasks are still processing on GEE servers. Run this cell again in a minute to refresh the status.
